# Acervo que Fala — Notebook 06: o lote de avaliação (40 casos + 10 de holdout)

**Projeto final** · Inteligência Artificial Generativa & Large Language Models (ICA/PUC-Rio) · Eduardo Tosto

Até aqui o sistema foi desenvolvido sobre um **lote de desenvolvimento** de 20 objetos: dez rodadas de geração, seis revisões editoriais e um redesenho do sistema de instruções. Este notebook faz a pergunta seguinte: **o sistema, como ficou, funciona em objetos que não participaram dos ajustes?**

**O que roda aqui.** O sistema congelado em 28/08/2026, sem alteração: observação v3.1, redação v13, rubrica v1.4, garantias em código e validador com um retry. Os blocos que definem o sistema são o **mesmo texto** do Notebook 04 v10 — copiados por script, não redigitados, e os três prompts foram conferidos contra os que ficaram gravados no resultado do lote v10. Muda só o que está em volta: a lista de objetos, o salvamento, e o registro de tempo e de erros.

**Os dois conjuntos.**

- **40 casos de avaliação** (`avaliacao/casos.jsonl`): escolhidos e anotados em 24/08/2026, **antes** de qualquer otimização de prompt, com critérios por caso e cobertura de 8 categorias de borda (foto parcial, artefato de estúdio, divergência entre foto e catálogo, jargão, metadado suspeito...).
- **10 casos de holdout** (`avaliacao/holdout.jsonl`): reservados na mesma data e nunca rodados. *Holdout* é o conjunto que fica fora de todo o desenvolvimento e roda **uma única vez**, no fim: a medida que não sofre influência de quem ajustou o sistema olhando resultados. Os dois conjuntos rodam na mesma sessão e são salvos em **arquivos separados**.

**Uma ressalva declarada.** Cinco dos 40 casos (Pote 9196, Faixa frontal 665, Flauta de pã 51023, Abano 63283 e Tanga 78838) são os objetos do *smoke test*: estiveram no lote de desenvolvimento, e o sistema foi ajustado olhando para eles. Ficam marcados no resultado (`visto_no_desenvolvimento`), e as métricas saem com e sem eles. Os outros 35 casos e os 10 de holdout nunca passaram pelo sistema.

**Garantias novas de execução** (não mudam o que o sistema escreve):

- **Checkpoint**: cada resposta do modelo é gravada no Drive assim que sai. Se a sessão do Colab cair, basta "Executar tudo" de novo — o notebook retoma de onde parou, sem refazer o que já foi gerado.
- **Erro gravado por item**: quando uma geração falha, o erro e a resposta bruta ficam no resultado. É a lição do bake-off, em que uma rodada inteira se perdeu porque o erro não era registrado.
- **Tempo medido**: segundos por item em cada etapa, gravados no resultado.

*Metodologia: projeto construído com LLMs como suporte (vibe coding) — cada célula explicada.*

### Como rodar
1. **Ambiente de execução → Alterar o tipo → GPU T4** · 2. **Executar tudo** · 3. Tempo estimado: **cerca de 2 h** para os 50 objetos (projeção a partir dos 45–55 min do lote de 20). Deixe a aba aberta. Se a sessão cair, repita o passo 2.

In [ ]:
# Etapa 1 — Instalação (Pillow travada, regra da casa) + checagem do ambiente
import PIL
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers pillow=={PIL.__version__}
import torch, transformers
from PIL import ImageDraw
from torchvision.io import decode_image
print(f"transformers {transformers.__version__} | GPU: {torch.cuda.is_available()}")
print("ambiente íntegro ✓")

## Etapa 2 — Buscar os 50 objetos, com salvaguardas de imagem (~7 min)

Os ids abaixo são os de `avaliacao/casos.jsonl` e `avaliacao/holdout.jsonl`, na ordem dos arquivos. Cada objeto vem direto da API pública do museu (Tainacan): a fotografia e o registro de catálogo.

As salvaguardas são as do Notebook 04:

- **Orientação EXIF** e conversão para **RGB**, para o modelo receber a foto como uma pessoa a vê no navegador;
- **Porteiro de resolução**: foto com menos de 100 mil pixels não gera descrição nenhuma — vira flag de dataset.

Uma proteção nova, porque agora são 150 pedidos à API em vez de 60: cada pedido tem até **3 tentativas**. Se as três falharem, o notebook **para** — caso de avaliação faltando não se pula em silêncio, porque mudaria o conjunto que está sendo medido.

In [ ]:
# Etapa 2 — Os 50 objetos do conjunto de avaliação, com as salvaguardas de imagem
import io, re, time, requests
from PIL import Image, ImageOps

BASE = "https://tainacan.museudoindio.gov.br/wp-json/tainacan/v2"
# avaliacao/casos.jsonl — 40 casos fixados em 24/08/2026, antes de qualquer otimização
IDS_CASOS = [9196, 680, 1260, 1826, 1996, 1045, 2151, 78838, 4286, 4116, 4211, 4246, 77963, 2601, 4056, 2826, 4606, 665, 2221, 2236, 2316, 1521, 1436, 5371, 1321, 5366, 1316, 1331, 63283, 3176, 500067, 5041, 777254, 4896, 775243, 51023, 205123, 204975, 205036, 206999]
# avaliacao/holdout.jsonl — 10 casos reservados na mesma data; rodam aqui pela única vez
IDS_HOLDOUT = [4391, 2746, 84836, 90371, 210599, 508608, 630, 1050, 499969, 199679]
# os 5 objetos do smoke test estiveram no lote de desenvolvimento (ressalva declarada)
IDS_VISTOS = {9196, 665, 51023, 63283, 78838}
assert len(IDS_CASOS) == 40 and len(IDS_HOLDOUT) == 10
assert not set(IDS_CASOS) & set(IDS_HOLDOUT), "um caso não pode estar nos dois conjuntos"

CAMPOS_REGISTRO = ["Nome do item", "Povo", "Categoria", "Matéria-prima",
                   "Técnica de confecção", "Dimensões", "Função",
                   "Estado de origem", "Ano de aquisição do objeto", "Descrição"]


def buscar(url, espera=60, tentativas=3):
    """Até 3 tentativas por pedido, com pausa crescente. Se as três falharem o notebook
    PARA: caso de avaliação faltando não se pula em silêncio."""
    for t in range(1, tentativas + 1):
        try:
            r = requests.get(url, timeout=espera)
            r.raise_for_status()
            return r
        except Exception as e:
            if t == tentativas:
                raise RuntimeError(f"a API do museu não respondeu ({url}): {e!r}")
            time.sleep(5 * t)


objetos = []
for item_id in IDS_CASOS + IDS_HOLDOUT:
    item = buscar(f"{BASE}/items/{item_id}").json()
    url_imagem = re.search(r'src="([^"]+)"', item["document_as_html"]).group(1)
    foto = Image.open(io.BytesIO(buscar(url_imagem, espera=90).content))
    foto = ImageOps.exif_transpose(foto).convert("RGB")  # salvaguardas
    meta_bruto = buscar(f"{BASE}/item/{item_id}/metadata").json()
    todos = {m["metadatum"]["name"]: m["value_as_string"] for m in meta_bruto if m.get("value_as_string")}
    registro = {c: todos.get(c, "") for c in CAMPOS_REGISTRO}
    # Porteiro de resolução (5ª adjudicação): foto pequena demais não gera descrição
    # nenhuma — vira flag de dataset.
    w, h = foto.size
    resolucao_ok = w * h >= 100_000
    objetos.append({"id": item_id, "titulo": item["title"], "foto": foto,
                    "registro": registro, "resolucao": f"{w}×{h}",
                    "resolucao_ok": resolucao_ok,
                    "conjunto": "holdout" if item_id in IDS_HOLDOUT else "casos",
                    "visto_no_desenvolvimento": item_id in IDS_VISTOS})
    alerta = "" if resolucao_ok else "  ⚠ SEM RESOLUÇÃO — não será descrito"
    print(f"✓ {objetos[-1]['conjunto']:7} {item_id} — {item['title']} ({registro['Povo']}) {w}×{h}px{alerta}")
print(f"{len(objetos)} objetos carregados | "
      f"{sum(1 for o in objetos if not o['resolucao_ok'])} sem resolução mínima")

## Etapa 3 — Rubrica v1.4 e o RAG híbrido

Célula idêntica à do Notebook 04 v10. A rubrica é a base de conhecimento do projeto (diretrizes por categoria de objeto + glossário). O **RAG** (*retrieval-augmented generation*: buscar trechos relevantes e entregá-los ao modelo junto com a tarefa) aqui é híbrido: a diretriz da categoria vem do campo `Categoria` do registro — garantia, não sorteio —, e o glossário vem da busca por similaridade semântica.

In [ ]:
# Etapa 3 — Rubrica v1.4 (ganhou 'gameliforme' no glossário) + RAG híbrido: categoria garantida + glossário recuperado
import json, os, requests
from google.colab import drive
from sentence_transformers import SentenceTransformer, util

drive.mount("/content/drive")
PROJETO = "/content/drive/MyDrive/00_IA/GenAI & LLMs - PUC/Projeto_LLM"
RUBRICA_DRIVE = f"{PROJETO}/dados/rubrica_v1_4.json"
RUBRICA_REPO = ("https://raw.githubusercontent.com/eduardotosto/acervo-que-fala/"
                "main/dados/rubrica/rubrica.json")
if os.path.exists(RUBRICA_DRIVE):
    with open(RUBRICA_DRIVE, encoding="utf-8") as f:
        rubrica = json.load(f)
    print(f"rubrica lida do Drive ✓")
else:
    rubrica = requests.get(RUBRICA_REPO, timeout=60).json()
    print("rubrica lida do repositório público (ainda não está no Drive) ✓")

trechos = rubrica["trechos"]
glossario = [t for t in trechos if t["categoria"] == "glossario"]
trechos_categoria = [t for t in trechos if t["categoria"] != "glossario"]
por_categoria = {}
for t in trechos_categoria:
    por_categoria.setdefault(t["categoria"], []).append(t)

# Embedder na CPU: 23 trechos e 20 consultas são trabalho trivial, e a T4 fica inteira
# para o modelo 4-bit (armadilha conhecida: device_map="auto" com a GPU já ocupada
# manda camadas para a CPU e quebra a geração).
embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device="cpu")
vet_glossario = embedder.encode([t["texto"] for t in glossario], convert_to_tensor=True)
vet_categoria = embedder.encode([t["texto"] for t in trechos_categoria], convert_to_tensor=True)
# A família Qwen3-Embedding é instruída: a CONSULTA vai com o prompt de query, os
# documentos vão sem. Se a versão instalada não trouxer o prompt, seguimos sem ele.
PROMPT_QUERY = "query" if "query" in (getattr(embedder, "prompts", None) or {}) else None

def recuperar(categoria, consulta, k_glossario=2, k_fallback=2):
    """RAG híbrido. A diretriz da categoria vem do REGISTRO (o campo Categoria existe em
    100% dos itens) — é garantia, não sorteio; o glossário vem da busca semântica, que é
    onde a recuperação realmente agrega. Categoria fora da rubrica cai no modo semântico."""
    v = embedder.encode(consulta, prompt_name=PROMPT_QUERY, convert_to_tensor=True)
    scores = util.cos_sim(v, vet_glossario)[0]
    achados = [glossario[i] for i in scores.argsort(descending=True)[:k_glossario].tolist()]
    fixos = por_categoria.get(categoria)
    if fixos is None:
        s2 = util.cos_sim(v, vet_categoria)[0]
        fixos = [trechos_categoria[i] for i in s2.argsort(descending=True)[:k_fallback].tolist()]
    return fixos + achados

print(f"rubrica {rubrica['versao']}: {len(por_categoria)} categorias + {len(glossario)} "
      f"trechos de glossário | prompt de query: {PROMPT_QUERY or 'indisponível'} ✓")

## Etapa 4 — O modelo e o teste de fumaça

O carregamento é idêntico ao do Notebook 04 v10: **Qwen3-VL-8B quantizado em 4-bit** (a quantização reduz a precisão dos números do modelo para ele caber na memória da T4 gratuita).

Depois do carregamento, duas coisas novas:

- as **versões das bibliotecas** ficam gravadas no resultado — o Colab instala sempre a versão mais recente, e uma rodada só se reproduz se o ambiente estiver registrado;
- um **teste de fumaça** (*smoke test*: a menor execução possível, só para saber se o caminho inteiro funciona): uma geração mínima, sem tratamento de erro. Se a geração estiver quebrada, o notebook quebra aqui, em segundos, e não depois de duas horas.

In [ ]:
# Etapa 4 — Modelo (Qwen3-VL-8B em 4-bit, como nos notebooks anteriores)
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODELO = "Qwen/Qwen3-VL-8B-Instruct"
quantizacao = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
modelo = Qwen3VLForConditionalGeneration.from_pretrained(
    MODELO, quantization_config=quantizacao, device_map="auto"
)
processador = AutoProcessor.from_pretrained(MODELO)

def gerar(conteudo, max_tokens=400):
    conversa = [{"role": "user", "content": conteudo}]
    entradas = processador.apply_chat_template(
        conversa, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(modelo.device)
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_tokens)
    return processador.decode(saida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def extrair_json(texto):
    texto = re.sub(r"^```(json)?|```$", "", texto.strip(), flags=re.MULTILINE).strip()
    inicio, fim = texto.find("{"), texto.rfind("}")
    return json.loads(texto[inicio:fim + 1])

print("modelo carregado ✓")

# ---- novo no Notebook 06: ambiente registrado + teste de fumaça ----
VERSOES_AMBIENTE = {"transformers": transformers.__version__, "torch": torch.__version__,
                    "pillow": PIL.__version__}
print("ambiente:", VERSOES_AMBIENTE)

# Teste de fumaça: geração mínima SEM try/except — se a geração estiver quebrada, o
# notebook quebra aqui, com o erro na tela, e não depois de duas horas de lote.
fumaca = gerar([{"type": "text", "text": 'Responda APENAS com este JSON: {"ok": true}'}], max_tokens=30)
print("fumaça:", repr(fumaca[:150]))
assert fumaca.strip(), "a geração devolveu vazio — investigar antes do lote"
extrair_json(fumaca)
print("fumaça: JSON extraído ✓")

## Etapa 4b — Checkpoint: nada do que o modelo gerou se perde

Uma rodada de 2 horas no Colab gratuito pode cair no meio. **Checkpoint** é o ponto de retomada: depois de cada resposta do modelo, o estado inteiro do lote é gravado num arquivo do Drive.

Ao rodar de novo, o notebook lê esse arquivo e devolve a cada objeto o que já tinha sido gerado. Só o que veio do modelo é recuperado do checkpoint; o que é cálculo (parse das seções, escala, flags de registro) é refeito, porque é determinístico e custa milissegundos.

Duas regras que protegem a medição:

- **Falha não ganha segunda chance.** Item cuja geração falhou fica registrado como falha e não é refeito na retomada — refazer só os que falharam daria a eles uma tentativa que os outros não tiveram.
- **Três falhas seguidas param o notebook.** Falha isolada é comportamento do modelo; três seguidas são ambiente quebrado (memória, biblioteca). Nesse caso as três saem do checkpoint e o notebook para com o erro na tela.

In [ ]:
# Etapa 4b — Checkpoint: o que o modelo gerou vai para o Drive assim que sai
import gc

VERSAO_NOTEBOOK = "06_lote_avaliacao"  # checkpoint de outra versão do notebook é ignorado
CHECKPOINT = f"{PROJETO}/resultados/06_lote_avaliacao_checkpoint.json"
os.makedirs(os.path.dirname(CHECKPOINT), exist_ok=True)

# só o que custa GPU volta do checkpoint; o que é cálculo se refaz a cada execução
CAMPOS_OBSERVACAO = ["observacao", "erro_observacao", "segundos_observacao"]
CAMPOS_CONTRADICAO = ["contradicao", "segundos_contradicao"]
CAMPOS_REDACAO = ["alt_bruto", "alt_text", "descricao_objeto", "flags", "retry", "json_valido",
                  "diretrizes_usadas", "resposta_bruta", "resposta_bruta_retry", "erro",
                  "segundos_redacao"]
CAMPOS_GERADOS = CAMPOS_OBSERVACAO + CAMPOS_CONTRADICAO + CAMPOS_REDACAO


def salvar_checkpoint():
    estado = {"versao": VERSAO_NOTEBOOK, "ids": [o["id"] for o in objetos],
              "itens": {str(o["id"]): {k: o[k] for k in CAMPOS_GERADOS if k in o}
                        for o in objetos}}
    provisorio = CHECKPOINT + ".tmp"
    with open(provisorio, "w", encoding="utf-8") as f:
        json.dump(estado, f, ensure_ascii=False)
    os.replace(provisorio, CHECKPOINT)  # troca de uma vez: o arquivo nunca fica pela metade


def carregar_checkpoint():
    if not os.path.exists(CHECKPOINT):
        return "nenhum checkpoint — rodada começa do zero"
    try:
        with open(CHECKPOINT, encoding="utf-8") as f:
            estado = json.load(f)
    except Exception as e:
        return f"checkpoint ilegível ({e!r}) — ignorado, rodada começa do zero"
    if estado.get("versao") != VERSAO_NOTEBOOK or estado.get("ids") != [o["id"] for o in objetos]:
        return "checkpoint de outra versão ou de outra lista de objetos — ignorado"
    for o in objetos:
        o.update(estado["itens"].get(str(o["id"]), {}))
    feitos = {nome: sum(1 for o in objetos if campo in o) for nome, campo in
              (("observações", "observacao"), ("contradições", "contradicao"),
               ("redações", "segundos_redacao"))}
    return f"checkpoint encontrado — retomando com {feitos}"


def liberar_memoria():
    gc.collect()
    torch.cuda.empty_cache()


def parar_se_quebrado(falhas, campos, etapa):
    """Três falhas SEGUIDAS não são acaso: são ambiente quebrado (memória, biblioteca).
    O notebook para, em vez de gastar duas horas produzindo erro. As três saem do
    checkpoint, para serem refeitas quando o ambiente estiver de pé."""
    if len(falhas) < 3:
        return
    ultimo = falhas[-1].get("erro") or falhas[-1].get("erro_observacao")
    for o in falhas:
        for c in campos:
            o.pop(c, None)
    salvar_checkpoint()
    raise RuntimeError(f"{etapa}: 3 falhas seguidas — último erro: {ultimo}")


print(carregar_checkpoint())

## Etapa 5 — Observação v3.1 e a varredura de artefatos (~40 min)

Prompt, varredura de artefatos, flags de cor e parse das seções são o **mesmo texto** do Notebook 04 v10. Nesta etapa o modelo recebe **só a fotografia** — é a separação que permite auditar depois se um erro nasceu no olho ou na escrita.

O laço ganhou o que a Etapa 4b descreve: pula o que o checkpoint já tem, grava o erro quando a geração falha e mede o tempo de cada item.

In [ ]:
import time
# Etapa 5 — Observação v3.1 (uma linha nova: TODAS as cores — resto inalterado).
# O bloco de varredura de artefatos é o mesmo texto de avaliacao/checar_lote.py.
import collections

PROMPT_OBSERVACAO_V3 = """Você está diante da fotografia de um objeto do acervo de um museu — objetos etnográficos de povos indígenas do Brasil, fotografados em estúdio. Este contexto serve para você reconhecer materiais e situações de estúdio; NÃO use para adivinhar o que o objeto é ou significa. Descreva somente o que está visível NESTA fotografia.

Preencha as seções abaixo, nesta ordem:

OBJETO: o que se vê, em uma frase — forma geral, sem nomear função nem significado.
MATERIAIS E CORES: os materiais aparentes e suas cores, do maior para o menor — liste TODAS as cores presentes, inclusive as minoritárias e as parcialmente encobertas. Material que não dá para identificar recebe o termo genérico ("fibra", "madeira clara") — nunca chute espécie ou origem.
PADRÕES E TEXTURAS: desenhos, tramas e acabamentos visíveis, descritos pela forma (linhas, xadrez, diagonais) — somente se existirem.
PARTES E QUANTIDADES: partes distinguíveis e contáveis (tubos, furos, alças, penas destacadas).
POSIÇÃO: como o objeto está na foto (de pé, deitado, inclinado) e partes internas visíveis (boca, interior, verso).
LEGIBILIDADE: o que estiver ilegível ou incerto — declare a incerteza em vez de estimar.
FUNDO E ESTÚDIO: o fundo e qualquer artefato de estúdio (etiqueta, numeração, cartela de cores, régua, suporte).
ENQUADRAMENTO: inteiro OU detalhe — "detalhe" SÓ se a foto mostra claramente apenas parte do objeto; objeto que encosta ou sangra nas margens conta como inteiro.
ARTEFATOS: os artefatos de estúdio vistos, separados por vírgula, ou "nenhum".

Regra geral: o que não está visível não existe para esta descrição. Responda em português."""

# ---- laço do Notebook 06: checkpoint, erro gravado, tempo medido ----
falhas_seguidas = []
for n, obj in enumerate(objetos, 1):
    if not obj["resolucao_ok"]:
        obj["observacao"] = ""
        print(f"[{n}/{len(objetos)}] {obj['titulo']} PULADO (sem resolução: {obj['resolucao']})")
        continue
    if "observacao" in obj:
        print(f"[{n}/{len(objetos)}] {obj['titulo']} já observado (checkpoint)")
        continue
    inicio = time.time()
    try:
        obj["observacao"] = gerar(
            [{"type": "image", "image": obj["foto"]}, {"type": "text", "text": PROMPT_OBSERVACAO_V3}],
            max_tokens=500,
        )
        falhas_seguidas = []
        print(f"[{n}/{len(objetos)}] {obj['titulo']} observado ✓")
    except Exception as e:
        obj["observacao"] = ""
        obj["erro_observacao"] = repr(e)
        falhas_seguidas.append(obj)
        print(f"[{n}/{len(objetos)}] {obj['titulo']} FALHOU na observação: {repr(e)[:90]}")
    obj["segundos_observacao"] = round(time.time() - inicio, 1)
    liberar_memoria()
    parar_se_quebrado(falhas_seguidas, CAMPOS_OBSERVACAO, "observação")
    salvar_checkpoint()

# A secao ARTEFATOS sozinha nao basta. Medido no lote v6: em 2 dos 4 itens com artefato
# real, o modelo descreveu o artefato na secao onde o viu (PARTES E QUANTIDADES,
# LEGIBILIDADE, FUNDO E ESTUDIO) e respondeu "nenhum" em ARTEFATOS - ele nao repete o
# que ja disse. A varredura le todas as secoes e descarta as mencoes sob negacao
# ("nao ha etiquetas", "sem numeracao"), que sao a maioria. No lote v6: 4/4 itens
# corretos, 0 falso positivo, contra 2/4 da secao sozinha.
# "suporte" ficou de fora: nomeia tanto a base de estudio quanto uma parte do proprio
# objeto (a peca central da bracadeira 1366) - ambiguidade que nao da para resolver aqui.
FAMILIAS_ARTEFATO = [("etiqueta", r"etiqueta\w*|r[óo]tulo\w*"),
                     ("inscrição", r"numera[çc]\w*|marca[çc][ãa]o\w*|inscri[çc]\w*"),
                     ("cartela de cores", r"cartela\w*|escala\s+de\s+cores?"),
                     ("régua", r"r[ée]gua\w*")]
RE_TERMO_ARTEFATO = re.compile(
    r"(?<![a-zà-ú])(" + "|".join(p for _, p in FAMILIAS_ARTEFATO) + ")", re.I)
RE_NEGACAO = re.compile(r"(?<![a-zà-ú])(n[ãa]o\s|nenhum\w*|sem\s|nada\s|aus[êe]ncia)", re.I)


def artefatos_da_observacao(obs):
    """Uma flag por familia de artefato citada na observacao fora de contexto de
    negacao, com a frase mais curta como detalhe."""
    por_familia = {}
    for m in RE_TERMO_ARTEFATO.finditer(obs):
        ini = max(obs.rfind(".", 0, m.start()), obs.rfind("\n", 0, m.start())) + 1
        if RE_NEGACAO.search(obs[ini:m.start()]):
            continue
        fim = obs.find(".", m.end())
        frase = re.sub(r"\s+", " ", obs[ini:fim if fim > 0 else m.end() + 60]).strip(" .;,")
        frase = re.sub(r"^[#*\s]*[A-ZÀ-Ú ]{4,}[#*\s]*:[#*\s]*", "", frase)
        nome = next(n for n, pat in FAMILIAS_ARTEFATO if re.match(pat, m.group(0), re.I))
        if nome not in por_familia or len(frase) < len(por_familia[nome]):
            por_familia[nome] = frase
    return [{"tipo": "artefato_estudio", "detalhe": f"{n}: {t}"} for n, t in por_familia.items()]

# Cor vista na foto que o registro nao nomeia. Reencontra automaticamente o achado
# fundador do projeto: as duas penas AZUIS da Faixa Kalapalo (665), que o registro
# nao menciona. So dispara quando o registro DESCREVE cores (registro que nao fala de
# cor nao autoriza divergencia) e so para cores informativas - bege, marrom, cinza e
# afins sao a cor natural do material, que o catalogo nunca nomeia.
CORES_INFORMATIVAS = {"azul": r"azu[lm]|azulad", "verde": r"verde|esverdead",
                      "vermelho": r"vermelh|avermelhad", "amarelo": r"amarel",
                      "laranja": r"laranj", "roxo": r"rox|violet", "rosa": r"rosa",
                      "preto": r"pret", "branco": r"branc"}
CORES_DE_MATERIAL = r"bege|marrom|castanh|cinza|creme|ocre|dourad|prate|amarronzad"


def cores_do_texto(texto):
    return {nome for nome, pat in CORES_INFORMATIVAS.items()
            if re.search(rf"(?<![a-zà-ú])(?:{pat})\w*", texto or "", re.I)}


def cores_divergentes(observacao, registro, secao_fn):
    """Cores que a observacao nomeia e o registro nao. Devolve lista de flags."""
    texto_reg = " ".join(str(registro.get(c, "")) for c in
                         ("Descrição", "Matéria-prima", "Técnica de confecção", "Nome do item"))
    no_registro = cores_do_texto(texto_reg)
    if not no_registro:
        return []
    # 6a adjudicacao (#6): cor de ARTEFATO (etiqueta branca, rotulo) nao e cor do
    # objeto — frases que citam artefato saem antes de extrair as cores
    texto_foto = (secao_fn(observacao, "MATERIAIS E CORES") + " " +
                  secao_fn(observacao, "PADRÕES E TEXTURAS"))
    texto_foto = " ".join(fr for fr in re.split(r"[.;]", texto_foto)
                          if not re.search(r"etiqueta|r[óo]tulo|cartela|marca[çc]|inscri[çc]", fr, re.I))
    na_foto = cores_do_texto(texto_foto)
    so_na_foto = sorted(na_foto - no_registro)
    if not so_na_foto:
        return []
    return [{"tipo": "divergencia_imagem_catalogo",
             "detalhe": f"a foto mostra {', '.join(so_na_foto)} e o registro nomeia só "
                        f"{', '.join(sorted(no_registro))}"}]

# --- Parse das seções ---
# O nome da seção volta com variações: no lote v6, 19 das 20 observações escreveram
# "FONDO E ESTÚDIO" e uma escreveu "FUNDOS E ESTÚDIO" — nenhuma acertou "FUNDO E
# ESTÚDIO". Com a grafia exigida exata, a seção não era lida NEM removida, e ninguém
# ficava sabendo. O padrão agora tolera a variação, e o relatório no fim da célula
# avisa quando um cabeçalho não aparece.
CABECALHO = (r"(?:OBJETO|MATERIAIS E CORES|PADRÕES E TEXTURAS|PARTES E QUANTIDADES|POSIÇÃO|"
             r"LEGIBILIDADE|F[OU]NDOS? E EST[ÚU]DIO|ENQUADRAMENTO|ARTEFATOS)")
CONSUMIDAS = r"(?:ENQUADRAMENTO|ARTEFATOS|F[OU]NDOS? E EST[ÚU]DIO)"
P = r"[#*]*\s*"

def secao(texto, nome):
    """Conteúdo de uma seção nomeada, ou string vazia. Aceita cabeçalho em negrito
    (**ENQUADRAMENTO:**) e em título markdown (### ENQUADRAMENTO:)."""
    m = re.search(rf"{nome}{P}:{P}(.+?)(?=\n{P}{CABECALHO}{P}:|\Z)", texto, re.S | re.I)
    return m.group(1).strip().strip("*#").strip() if m else ""

faltando = collections.Counter()
for obj in objetos:
    obs = obj["observacao"]
    if not obs:  # sem resolução: campos vazios, flag entra na redação
        obj.update(enquadramento="", enquadramento_ok=True, artefatos_secao=[],
                   flags_artefato=[], artefatos_obs=[], observacao_para_redacao="",
                   consulta_rag=obj["titulo"], marca_atribuicao="", flags_cor=[])
        continue
    enq = secao(obs, "ENQUADRAMENTO").lower()
    obj["enquadramento"] = "detalhe" if enq.startswith("detalhe") else "inteiro"
    obj["enquadramento_ok"] = enq.startswith(("inteiro", "detalhe"))
    # o que a seção ARTEFATOS respondeu, guardado só para medir a diferença
    art = secao(obs, "ARTEFATOS")
    obj["artefatos_secao"] = [] if (not art or art.lower().startswith("nenhum")) else [
        a.strip(" .") for a in art.split(",") if a.strip(" .")]
    # a flag vem da VARREDURA de todas as seções, não da obediência a uma delas
    obj["flags_artefato"] = artefatos_da_observacao(obs)
    obj["artefatos_obs"] = [f["detalhe"] for f in obj["flags_artefato"]]
    for nome in ("OBJETO", "MATERIAIS E CORES", "LEGIBILIDADE", "F[OU]NDOS? E EST[ÚU]DIO",
                 "ENQUADRAMENTO", "ARTEFATOS"):
        if not secao(obs, nome):
            faltando[nome] += 1
    # A redação recebe a observação sem as três seções que o código já consumiu.
    obj["observacao_para_redacao"] = re.sub(
        rf"\n?{P}{CONSUMIDAS}{P}:.*?(?=\n{P}{CABECALHO}{P}:|\Z)", "", obs, flags=re.S | re.I).strip()
    obj["consulta_rag"] = (f"{obj['titulo']}. {secao(obs, 'OBJETO')} "
                           f"{secao(obs, 'PADRÕES E TEXTURAS')}")[:400]

ok = sum(1 for o in objetos if o["enquadramento_ok"])
so_varredura = sum(1 for o in objetos if o["flags_artefato"] and not o["artefatos_secao"])
vazou = sum(1 for o in objetos if re.search(r"(?<![a-zà-ú])fundo\b", o["observacao_para_redacao"], re.I))
print(f"\nparse: {ok}/{len(objetos)} com ENQUADRAMENTO válido | "
      f"{sum(1 for o in objetos if o['enquadramento'] == 'detalhe')} 'detalhe'")
print(f"artefatos: {sum(len(o['flags_artefato']) for o in objetos)} flags pela varredura, "
      f"{sum(len(o['artefatos_secao']) for o in objetos)} pela seção ARTEFATOS "
      f"({so_varredura} itens que só a varredura pegou)")
print(f"seções ausentes por cabeçalho: {dict(faltando) or 'nenhuma'} | "
      f"{vazou} observações ainda citam 'fundo' depois da limpeza")

# A fórmula de atribuição vem sorteada pelo CÓDIGO, não pedida ao modelo. No lote v6,
# "segundo o registro do museu" abriu 20 de 20 textos: é o primeiro exemplo do prompt,
# copiado literalmente — o mecanismo do papagaio de exemplo. Pedir "formulação variada"
# não funciona: o modelo não sabe o que escreveu no item anterior. O código sabe.
FORMULAS_ATRIBUICAO = [
    "segundo o registro do museu",
    "o registro do museu informa que",
    "conforme o catálogo do museu",
    "de acordo com a ficha do museu",
    "o catálogo do museu registra que",
]
for n, obj in enumerate(objetos):
    obj["marca_atribuicao"] = FORMULAS_ATRIBUICAO[n % len(FORMULAS_ATRIBUICAO)]
    # cor vista na foto que o registro não nomeia — recupera sozinha o achado fundador
    # do projeto (as duas penas azuis da Faixa Kalapalo, ausentes do registro)
    obj["flags_cor"] = cores_divergentes(obj["observacao"], obj["registro"], secao)

print(f"cores: {sum(1 for o in objetos if o['flags_cor'])} itens com cor que o registro não nomeia "
      f"| atribuição: {len(FORMULAS_ATRIBUICAO)} fórmulas em rodízio")

## Etapa 5b — O que o código extrai do registro (~10 min)

Bloco idêntico ao do Notebook 04 v10. Aritmética e comparação não se pedem a modelo: a **escala** (maior dimensão, com o número literal do catálogo), o **teto de plausibilidade** por categoria, a **contagem** de partes e a **quarentena** (informação com flag não entra no texto) saem prontas do código.

A única chamada ao modelo aqui é a pergunta isolada sobre contradição entre campos do registro — e é ela que entra no checkpoint.

In [ ]:
# Etapa 5b — o que o CÓDIGO extrai do registro (aritmética não se pede a modelo).
# Este bloco é o mesmo texto de avaliacao/checar_lote.py, para que o notebook e a
# ferramenta que mede os lotes antigos nunca divirjam. Nesta versão ele carrega as
# decisões da 5ª adjudicação: número LITERAL do catálogo (41,5 não vira 42), medida com
# cordel/alça fora da escala, e QUARENTENA — flag sobre a dimensão tira a medida do
# texto; miniatura×função de uso real vira flag de contradição sem depender do modelo.
import collections

ROTULOS = r"(comprimento|altura|largura|di[âa]metro|espessura|profundidade)"
RE_ROTULO, RE_NUM = re.compile(ROTULOS, re.I), re.compile(r"\d+(?:[.,]\d+)?")
# medida "com cordel / com a alça esticada" mede o objeto pendurado, não a peça
RE_COM_CORDA = re.compile(
    r"com\s+(a\s+|o\s+)?(cordel|cord[aã]o|al[çc]a|amarra)|esticad|\+\s*(cordel|cord[aã]o|al[çc]a)", re.I)

# Teto de plausibilidade por categoria = Q3 + 3xIQR das dimensões do PRÓPRIO acervo
# (547 dos 555 itens de dados/itens.json têm medida parseável), com piso de 150 cm —
# abaixo disso, peça grande é plausível em qualquer categoria. No acervo inteiro o teto
# dispara 2 vezes: o abano de 290 cm (dimensão improvável, caso conhecido do projeto) e
# a capa de pele de onça de 223 cm (peça genuinamente grande). Flag é pedido de
# conferência humana, não veredito — a taxa de alarme é o que importa, e é de 0,4%.
TETOS_CATEGORIA = {
    "Adornos Plumários": 156,
    "Adornos de Materiais Ecléticos, Indumentária e Toucador": 106,
    "Armas": 491,
    "Cerâmica": 56,
    "Cordões e Tecidos": 103,
    "Etnobotânica": 33,
    "Instrumentos musicais e de sinalização": 76,
    "Objetos rituais, mágicos e lúdicos": 144,
    "Trançados": 181,
    "Utensílios e implementos de materiais ecléticos": 75,
}
PISO_SUSPEITA = 150
# miniatura: peça bem menor que o comum da categoria, em categorias de objeto grande
MEDIANAS_CATEGORIA = {"Cerâmica": 14.0, "Trançados": 41.5, "Armas": 212.5,
                      "Instrumentos musicais e de sinalização": 41.0,
                      "Utensílios e implementos de materiais ecléticos": 19.5}

def escala_do_registro(dimensoes):
    """Maior dimensão do objeto, em cm, com o rótulo — a regra editorial 25 ("escala é a
    maior dimensão, nunca a medida de uma parte"), resolvida em código. Trata a lista
    enumerada ("29,5; 21,5; ... e 9,5 cm de comprimento") herdando o rótulo do segmento
    seguinte. Devolve (valor, rótulo) ou None."""
    if not dimensoes or "cm" not in dimensoes.lower():
        return None
    segmentos = [s for s in re.split(r";|\s-\s", dimensoes) if s.strip()]
    candidatos = []
    for i, seg in enumerate(segmentos):
        if RE_COM_CORDA.search(seg):
            continue
        rot = RE_ROTULO.search(seg) or next(
            (RE_ROTULO.search(s) for s in segmentos[i + 1:] if RE_ROTULO.search(s)), None)
        rotulo = rot.group(1).lower() if rot else ""
        candidatos += [(float(m.group(0).replace(",", ".")), rotulo) for m in RE_NUM.finditer(seg)]
    return max(candidatos) if candidatos else None


def numero_pt(v):
    """O número exatamente como o catálogo o dá, em grafia brasileira — arredondar cria
    um número que não existe no registro (5ª adjudicação: 'cerca de 42' para 41,5)."""
    return f"{v:g}".replace(".", ",")


def analisar_registro(registro):
    """Devolve a linha ESCALA pronta para o prompt + as flags de metadado que a
    aritmética já resolve (dimensão fora do teto da categoria, ano impossível)."""
    cat, flags = registro.get("Categoria", ""), []
    e = escala_do_registro(registro.get("Dimensões", ""))
    if not e:
        escala = "não informada no registro — não escreva medida nenhuma"
    else:
        valor, rotulo = e
        escala = f"cerca de {numero_pt(valor)} cm" + (f" de {rotulo}" if rotulo else "")
        mediana = MEDIANAS_CATEGORIA.get(cat)
        if mediana and valor < 10 and valor < mediana / 2:
            # 4a revisao: dimensao atipicamente pequena pede conferencia humana, pelo
            # mesmo motivo do abano de 290 cm — miniatura genuina ou erro de registro
            flags.append({"tipo": "metadado_suspeito", "detalhe":
                          f"dimensão atipicamente pequena para {cat} ({numero_pt(valor)} cm; "
                          f"mediana da categoria {numero_pt(mediana)} cm) — miniatura genuína "
                          f"ou erro de registro; conferir"})
        teto = TETOS_CATEGORIA.get(cat)
        if teto and valor > max(teto, PISO_SUSPEITA):
            flags.append({"tipo": "metadado_suspeito", "detalhe":
                          f"{numero_pt(valor)} cm de {rotulo or 'dimensão'} está acima do teto de "
                          f"plausibilidade da categoria {cat} ({teto} cm, calculado do acervo)"})
    if re.search(r"al[çc]as? soltas?", registro.get("Descrição", "") or "", re.I) and e:
        flags.append({"tipo": "metadado_suspeito", "detalhe":
                      "a Descrição menciona alças soltas — o comprimento pode incluí-las; conferir"})
    desc = registro.get("Descrição", "") or ""
    if re.search(r"(?<![a-zà-ú])miniatura", desc, re.I) and re.search(
            r"ca[çc]a|pesca|ataque|guerra|defesa", registro.get("Função", "") or "", re.I):
        flags.append({"tipo": "metadado_suspeito", "detalhe":
                      "a Descrição diz miniatura/brinquedo e a Função descreve uso real "
                      "(caça, ataque...) — contradição entre campos do registro; conferir"})
    ano = (registro.get("Ano de aquisição do objeto") or "").strip()
    if ano.isdigit() and not (1850 <= int(ano) <= 2026):
        flags.append({"tipo": "metadado_suspeito", "detalhe": f"ano de aquisição improvável: {ano}"})
    # 5a adjudicacao: informacao flagada nao entra no texto. Qualquer flag sobre a
    # dimensao (teto, piso, alcas soltas) poe a medida em quarentena.
    if any(re.search(r"cm|alças soltas|dimensão", f["detalhe"]) for f in flags):
        escala = ("EM QUARENTENA — a medida do registro está sob conferência (flag): "
                  "não escreva medida nenhuma")
    return escala, flags

NUMEROS_PT = {"um": 1, "uma": 1, "dois": 2, "duas": 2, "três": 3, "tres": 3,
              "quatro": 4, "cinco": 5, "seis": 6, "sete": 7, "oito": 8, "nove": 9,
              "dez": 10, "onze": 11, "doze": 12}
RE_NUM_SUBST = re.compile(r"(?<![a-zà-ú])(" + "|".join(NUMEROS_PT) + r"|[2-9]|1[0-2])\s+([a-zà-ú]{3,}s)(?![a-zà-ú])", re.I)


def contagens(texto):
    """Pares (substantivo plural, número) escritos num texto: 'seis tubos' -> (tubos, 6)."""
    pares = {}
    for m in RE_NUM_SUBST.finditer(texto or ""):
        n = NUMEROS_PT.get(m.group(1).lower()) or int(m.group(1))
        pares.setdefault(m.group(2).lower(), n)
    return pares

# Contradição entre campos do registro: pergunta ISOLADA, fora da tarefa de escrita.
# (A heurística miniatura×função acima já cobre o caso conhecido em código; a pergunta
# continua para contradições que a heurística não enxerga.)
PROMPT_CONTRADICAO = """Leia os campos do registro de catálogo de um museu abaixo e responda se existe CONTRADIÇÃO INTERNA: dois campos que afirmam coisas incompatíveis sobre o MESMO objeto.

Exemplo de contradição: o campo Descrição diz "brinquedo em miniatura" e o campo Função diz "utilizado para caça" — o mesmo objeto não é as duas coisas.
Não é contradição: campo vazio, informação que falta, detalhe que só um campo traz, ou estilo de escrita.

REGISTRO:
{registro}

Responda APENAS com JSON: {{"contradicao": true ou false, "campos": ["campo A", "campo B"], "detalhe": "uma frase explicando"}}"""

# ---- laço do Notebook 06: a resposta do modelo entra no checkpoint ----
for n, obj in enumerate(objetos, 1):
    registro_txt = "\n".join(f"{k}: {v}" for k, v in obj["registro"].items() if v)
    obj["escala"], obj["flags_registro"] = analisar_registro(obj["registro"])
    # contagens do catálogo ("seis tubos") viajam prontas — prevalecem sobre a visual
    pares = contagens(obj["registro"].get("Descrição", ""))
    obj["contagem_registro"] = ("; ".join(f"{v} {k}" for k, v in pares.items())
                                or "nenhuma no registro")
    ja_flagado = any("miniatura" in f["detalhe"] for f in obj["flags_registro"])
    if "contradicao" not in obj:  # pergunta já respondida (checkpoint) não se refaz
        inicio = time.time()
        try:
            obj["contradicao"] = extrair_json(gerar(
                [{"type": "text", "text": PROMPT_CONTRADICAO.format(registro=registro_txt)}],
                max_tokens=250))
        except Exception:
            obj["contradicao"] = None
        obj["segundos_contradicao"] = round(time.time() - inicio, 1)
        liberar_memoria()
        salvar_checkpoint()
    r = obj["contradicao"]
    if r and r.get("contradicao") and r.get("detalhe") and not ja_flagado:
        obj["flags_registro"].append({"tipo": "metadado_suspeito", "detalhe":
                                      f"contradição no registro: {r['detalhe']}"})
    marca = f" | {len(obj['flags_registro'])} flag(s)" if obj["flags_registro"] else ""
    print(f"[{n}/{len(objetos)}] {obj['id']} escala: {obj['escala'][:52]}{marca}")

print()
print(f"{sum(len(o['flags_registro']) for o in objetos)} flags de metadado geradas pelo código; "
      f"{sum(1 for o in objetos if 'QUARENTENA' in o['escala'])} medidas em quarentena")

## Etapa 6 — Redação v13, com validação e um retry (~65 min)

Prompt, pós-processamento, quarentena e validador são o **mesmo texto** do Notebook 04 v10. Nesta etapa o modelo recebe **só texto**: a observação (sem as seções que o código já consumiu), o registro, a escala calculada, as contagens, a marca de atribuição sorteada e as diretrizes da rubrica.

O código confere o rascunho e, se ele falha, devolve o diagnóstico ao modelo **uma vez**. O laço grava, além do texto final, a resposta bruta do modelo e o erro quando há falha.

In [ ]:
PROMPT_REDACAO_V13 = """Você escreve descrições de acessibilidade para o acervo digital de um museu. Elas serão OUVIDAS por pessoas cegas, através de leitores de tela — escreva em linguagem cotidiana, com frases que funcionam no ouvido (ordem direta, sem parênteses longos).

INSUMOS — cada um autoriza um tipo de informação:

OBSERVAÇÃO VISUAL DA FOTOGRAFIA (autoriza: aparência — o que é visível):
{observacao}

ENQUADRAMENTO DECIDIDO NA OBSERVAÇÃO: {enquadramento}

ESCALA CALCULADA DO REGISTRO: {escala}

CONTAGENS DO REGISTRO (prevalecem SEMPRE sobre a contagem visual): {contagem}

MARCA DE ATRIBUIÇÃO DESTE TEXTO (use esta fórmula, exatamente): {marca}

EM QUARENTENA — informação flagada para revisão humana. NÃO escreva nada sobre:
{quarentena}

REGISTRO DO MUSEU (autoriza: fatos — sempre com atribuição; o título nomeia o objeto):
{registro}

DIRETRIZES PARA ESTE TIPO DE OBJETO (autorizam: vocabulário e o que observar nesta categoria):
{diretrizes}

CONTRATO DE FONTES — prevalece sobre qualquer outra regra:
1. Cada informação escrita precisa de fonte: visível na observação, escrita no registro (e então leva atribuição) ou vocabulário das diretrizes. Sem fonte, não escreva — omitir é sempre permitido; um texto curto e todo verificável vale mais que um completo com um palpite.
2. O CATÁLOGO MANDA NAS PALAVRAS: termo do registro que não é técnico se usa como está (cabo, pá, algodão, "dobradura afunilada") — não invente sinônimo nem nova forma de dizer o que o catálogo já diz bem; descrição visual do registro em linguagem comum (ex.: figuras em "X") entra com as palavras dele. Termo técnico do registro se traduz pelo glossário das diretrizes ("motivos zoomorfos" vira "figuras de animais"; "borda extrovertida" vira "boca que se abre para fora"); espécie de animal ou planta SÓ se o registro nomear — nunca pela aparência.
3. Incerteza se herda: o que estiver na seção LEGIBILIDADE da observação, ou vier com hesitação ("parece", "talvez", "possivelmente"), sai do texto ou vira o termo genérico — nunca vira afirmação.
4. Divergência não se resolve no texto: quando observação e registro conflitam, não escolha um lado nem harmonize — registre em flags como divergencia_imagem_catalogo e NÃO escreva a informação divergente. Quantidade: use SEMPRE o número de CONTAGENS DO REGISTRO; cores: use as que o REGISTRO nomeia (cor vista que o registro não nomeia já está em quarentena via flag).
5. Os exemplos deste prompt mostram a FORMA das frases, com lacunas [assim]; preencha sempre com o conteúdo deste objeto, nunca com as palavras do exemplo.

PRODUZA TRÊS SAÍDAS:

A) alt_text — o que a fotografia mostra, para quem não a vê.
   Uma frase, no máximo 30 palavras — o limite de 30 VENCE qualquer outra exigência desta seção: se faltar espaço, corte adjetivo e detalhe secundário, nunca o objeto e o povo.
   Começa pelo objeto (nomeado pelo TÍTULO do registro) e pelo povo, com o nome do povo em maiúscula inicial.
   Contém: o material; as cores VISÍVEIS na observação que o registro confirma, onde a cor informa — elemento ou cor que só existe no catálogo e NÃO aparece na foto (decoração encoberta, roseta em foto de detalhe) nunca entra no alt, só na descrição; a forma dos padrões, sempre pela geometria OU pelo termo que a diretriz/glossário nomear (espinha-de-peixe, gregas) — quando a diretriz nomeia o padrão que você vê, USE esse termo; NUNCA descreva padrão por semelhança com letra, figura ou objeto ("em forma de G", "como uma flor", "lembra uma coroa" são erros) — exceto quando o próprio registro usar a comparação (figuras em "X").
   MATERIAL NATURAL NÃO TEM NOME DE COR: madeira, fibra, argila, algodão, couro e afins são no máximo "clara" ou "escura" — nunca bege, marrom ou castanho; a procedência (tipo de madeira, de fibra) entra quando o catálogo a dá. Cores nomeadas só em pintura, tingimento, penas e miçangas. Construção: miçangas são "confeccionadas com [linha]" e peças plumárias são "compostas por penas..." — nunca "[material] sobre [material]". Quando a peça tem pintura sobre a base, primeiro a pintura e suas cores, depois a base sem cor ("pintura em vermelho e preto sobre a cerâmica"). Cada cor fica PRESA à parte que a exibe ("cabo escuro, bocal claro") — enumerar cores soltas perde a referência.
   Se o ENQUADRAMENTO diz "detalhe", comece com "Detalhe de [objeto]"; se diz "inteiro", não mencione enquadramento nem orientação.
   Aves de penas: só as que o registro nomear, com as cores QUE O REGISTRO dá a elas: "penas [cores do registro] de [ave]".
   Não aparecem aqui: fundo ou estúdio, artefato de inventário, medida, alça/cordel/corda (salvo se for a característica principal do objeto), frases de ausência ("sem X", "não há X") e palavras de seção da observação ("partes contáveis", "legível").

B) descricao_objeto — o objeto em si, para quem quer conhecê-lo além da foto. Dois parágrafos, no máximo 120 palavras somadas.
   1º: a PRIMEIRA PALAVRA é o nome do objeto — sem "O objeto é…", "Trata-se de…", "é um objeto". A função entra nessa primeira frase SÓ quando diz algo que o nome não diz (caça, ritual, preparo) — bolsa que guarda, panela que cozinha, remo que rema e pulseira no pulso NÃO se explicam. Depois, a aparência: formas, materiais e padrões, cada informação uma vez.
   2º: os fatos do catálogo, AGRUPADOS, em frases naturais: a marca de atribuição aparece UMA vez, ao entrar neles, com a fórmula exata de MARCA DE ATRIBUIÇÃO — o texto PRECISA conter essa fórmula, e a frase que a contém começa com letra MAIÚSCULA. O povo, o ano de aquisição e o estado de origem SEMPRE entram quando o registro os tem, e o texto nunca termina na marca — os fatos vêm depois dela. A escala é EXATAMENTE a de ESCALA CALCULADA DO REGISTRO, em frase natural — não recalcule, não escolha outra medida, não cite medida de parte nem de alça; se lá diz para não escrever medida, NÃO escreva.
   Este texto descreve o objeto, não a fotografia: posição, fundo, enquadramento, "visível/parcialmente visível" não existem aqui; relações de ponto de vista viram relações da peça ("decrescentes", "em degraus"). Em amostras (argila, resina), o CONTENEDOR (tubo, tampa, frasco) só existe no alt — a descrição é do conteúdo, com o tipo e a procedência que o catálogo der. Não repita frases do alt nem a mesma palavra várias vezes.
   O texto diz o que existe: nada de "sem [coisa]", "não há [coisa]"; o que o registro não traz não vira frase.

C) flags — o que precisa de revisão humana. Artefatos, metadados improváveis e cores divergentes já foram registrados pelo código; concentre-se no que só quem leu os dois lados percebe:
   - divergencia_imagem_catalogo: conflito entre observação e registro (quantidade, material, formato) — ou objeto visto diferente do que o título nomeia (use o título no texto e registre aqui).
   Lista vazia [] se não houver nada.

Responda APENAS com JSON: {{"alt_text": "...", "descricao_objeto": "...", "flags": [{{"tipo": "...", "detalhe": "..."}}]}}"""

# Pós-processamento conservador: remove o fundo de estúdio residual SÓ quando o trecho
# termina em pontuação dentro de no máximo duas palavras ("..., sobre fundo bege." ✓).
# Em "sobre fundo bege e boca larga" ele não casa e não remove nada — amputar a frase
# seria pior que deixar passar, e o que sobra a verificação acusa.
RE_FUNDO = re.compile(
    r",?\s*\b(?:sobre|em|contra|com|num|no|sob)\s+(?:um |uma |o |a )?fundo\b"
    r"(?:\s+[a-zà-ú-]+){0,2}\s*(?=[,.;]|$)", re.I)

def capitalizar_frases(texto):
    """6a adjudicacao: a marca de atribuicao em minuscula abria frases sem maiuscula.
    Tipografia e deterministica — corrigir em codigo, nao pedir ao modelo."""
    return re.sub(r"(^|[.!?]\s+)([a-zà-ú])", lambda m: m.group(1) + m.group(2).upper(), texto or "")


def pos_processar_alt(alt):
    alt = RE_FUNDO.sub("", alt)
    alt = re.sub(r",\s*,", ",", alt)
    alt = re.sub(r"\s{2,}", " ", alt).strip(" ,")
    if alt and not alt.endswith("."):
        alt += "."
    return capitalizar_frases(alt)

# 6a adjudicacao: material natural nao tem nome de cor (mesmo padrao da regua)
RE_COR_MATERIAL = re.compile(
    r"(?<![a-zà-ú])(?:(?:base|superfície|argila|cerâmic\w+|madeira|fibra\w*|palha|couro|"
    r"osso|algodão|linha|fio|fios|taquara|cabaça|tucum|buriti|barro|vime|folha)\s+"
    r"(?:de\s+\w+\s+)?(?:bege|marro[mn]|castanh\w+|creme|pard[ao]s?|amarronzad\w+|"
    r"acinzentad\w+|terros[ao]s?)"
    r"|tonalidades?\s+(?:bege|marro[mn]|castanh\w+|creme|pard[ao]s?)"
    r"|(?:bege|marro[mn])[\w-]*[\s-]*(?:acinzentad|clar|escur)\w*\s+(?:e\s+bege)?)", re.I)

# ---- Quarentena: a lista do que as flags tiraram do texto (5ª adjudicação) ----
def montar_quarentena(obj):
    itens = []
    for f in obj["flags_cor"]:
        m = re.search(r"a foto mostra ([^e]+?) e o registro", f["detalhe"])
        if m:
            itens.append(f"as cores {m.group(1).strip()} (vistas na foto, ausentes do registro)")
    for f in obj["flags_registro"]:
        det = f["detalhe"]
        if "miniatura/brinquedo" in det or "contradição" in det:
            itens.append("a função do registro e a palavra miniatura (contradição flagada)")
        elif re.search(r"cm|dimensão|alças", det):
            itens.append("qualquer medida ou tamanho")
    if not obj["resolucao_ok"]:
        itens.append("TUDO — sem resolução de imagem, não há descrição")
    return chr(10).join(f"- {x}" for x in itens) if itens else "- nada em quarentena"

# ---- Validação + 1 retry: as exigências que o 8B costuma perder de primeira ----
def validar_rascunho(obj, alt, descricao):
    erros = []
    if len(alt.split()) > 30:
        erros.append(f"o alt tem {len(alt.split())} palavras — corte para no máximo 30")
    if obj["marca_atribuicao"] and obj["marca_atribuicao"] not in descricao.lower():
        erros.append(f"a descrição precisa conter a fórmula exata: {obj['marca_atribuicao']}")
    for texto, nome in ((alt, "alt_text"), (descricao, "descricao_objeto")):
        m = re.search(r"(?<![a-zà-ú])(sem [a-zà-ú]+|não há)", texto, re.I)
        if m:
            erros.append(f"remova a frase de ausência ('{m.group(0)}') do {nome}")
    if "QUARENTENA" in obj["escala"] and re.search(r"\d+[.,]?\d*\s*cm", descricao):
        erros.append("a medida está em quarentena — remova toda medida da descrição")
    # os tres residuos que sobraram no lote v8 (o prompt pede e o 8B ignora; o retry cobra):
    m_j = re.search(r"globular|extrovertid\w*|reticulad\w*|zoomorf\w*", alt + " " + descricao, re.I)
    if m_j:
        erros.append(f"'{m_j.group(0)}' é jargão de catálogo — traduza: globular = de corpo "
                     f"arredondado; borda extrovertida = boca que se abre para fora; "
                     f"motivos zoomorfos = figuras de animais")
    povo_v = (obj["registro"].get("Povo") or "").strip()
    if povo_v and povo_v.split()[0].lower() not in alt.lower():
        erros.append(f"o alt_text precisa citar o povo {povo_v.split()[0]}")
    m_f = re.search(r"(flauta[^.]{0,45}instrumento music|remo[^.]{0,50}(desloc|remar|vias aquáticas)"
                    r"|panela[^.]{0,50}(cozinhar|servir)|bolsa[^.]{0,45}(guardar|transportar)"
                    r"|pulseira[^.]{0,45}pulso)", (alt + " " + descricao).lower())
    if m_f and not (obj["registro"].get("Função") or "").strip():
        # 6a adjudicacao: funcao descrita no campo Função pode entrar — catalogo manda
        erros.append(f"função óbvia ('{m_f.group(0)[:40]}') — sem fonte no catálogo; corte a explicação")
    # 6a adjudicacao: material natural nao tem nome de cor
    m_cm = RE_COR_MATERIAL.search(alt + " " + descricao)
    if m_cm:
        erros.append(f"'{m_cm.group(0)[:30]}' dá nome de cor a material natural — no máximo clara/escura; "
                     f"cor nomeada só em pintura, tingimento, penas e miçangas")
    if povo_v and povo_v.split()[0].lower() not in descricao.lower():
        erros.append(f"a descrição precisa citar o povo {povo_v.split()[0]}")
    ano_v = (obj["registro"].get("Ano de aquisição do objeto") or "").strip()
    if ano_v.isdigit() and ano_v not in descricao:
        erros.append(f"a descrição precisa citar o ano de aquisição ({ano_v}) — o registro o tem")
    origem_v = (obj["registro"].get("Estado de origem") or "").strip()
    if origem_v and origem_v.split()[0].lower() not in descricao.lower():
        erros.append(f"a descrição precisa citar a origem ({origem_v}) — o registro a tem")
    if not descricao.rstrip().endswith("."):
        erros.append("a descrição terminou sem ponto final — complete os fatos depois da marca de atribuição")
    pal_alt = alt.lower().split()
    for i5 in range(len(pal_alt) - 4):
        tr5 = " ".join(pal_alt[i5:i5 + 5])
        if len(tr5) > 18 and tr5 in descricao.lower():
            erros.append(f"a descrição repete literalmente o alt ('{tr5[:36]}...') — reformule")
            break
    if obj["registro"].get("Categoria") == "Etnobotânica" and re.search(
            r"tubo|tampa|frasco|recipiente|vidro", descricao, re.I):
        erros.append("amostra: o contenedor (tubo/tampa) fica só no alt — descreva o conteúdo, "
                     "com tipo e procedência do catálogo")
    return erros

# ---- laço do Notebook 06: checkpoint, resposta bruta e erro gravados, tempo medido ----
falhas_seguidas = []
for n, obj in enumerate(objetos, 1):
    if not obj["resolucao_ok"]:
        obj.update(alt_bruto="", alt_text="", descricao_objeto="", diretrizes_usadas=[],
                   retry=False, json_valido=True,
                   flags=[{"tipo": "falta_de_resolucao",
                           "detalhe": f"imagem de {obj['resolucao']} px — sem resolução para "
                                      f"descrever; item devolvido ao dataset"}] + obj["flags_registro"])
        print(f"[{n}/{len(objetos)}] {obj['titulo']}: SEM RESOLUÇÃO — flag, nenhum texto")
        continue
    if obj.get("erro_observacao"):
        # sem observação não há o que redigir: o item fica registrado como falha
        obj.update(alt_bruto="", alt_text="", descricao_objeto="", diretrizes_usadas=[],
                   retry=False, json_valido=False, erro=obj["erro_observacao"],
                   flags=obj["flags_registro"])
        print(f"[{n}/{len(objetos)}] {obj['titulo']}: SEM OBSERVAÇÃO — falha registrada, nenhum texto")
        continue
    if "segundos_redacao" in obj:
        print(f"[{n}/{len(objetos)}] {obj['titulo']} já redigido (checkpoint)")
        continue
    registro_txt = chr(10).join(f"{k}: {v}" for k, v in obj["registro"].items() if v)
    achados = recuperar(obj["registro"]["Categoria"], obj["consulta_rag"])
    obj["diretrizes_usadas"] = [t["id"] for t in achados]
    prompt = PROMPT_REDACAO_V13.format(
        observacao=obj["observacao_para_redacao"], enquadramento=obj["enquadramento"],
        escala=obj["escala"], contagem=obj["contagem_registro"],
        marca=obj["marca_atribuicao"], quarentena=montar_quarentena(obj),
        registro=registro_txt, diretrizes=chr(10).join(f"- {t['texto']}" for t in achados),
    )
    obj["retry"] = False
    inicio = time.time()
    try:
        resposta = gerar([{"type": "text", "text": prompt}], max_tokens=700)
        obj["resposta_bruta"] = resposta
        saida = extrair_json(resposta)
        erros = validar_rascunho(obj, saida["alt_text"], saida["descricao_objeto"])
        if erros:
            # 1 retry com o diagnóstico — validador-e-reescrita, limitado a uma rodada
            obj["retry"] = True
            correcao = (prompt + chr(10) + chr(10) + "SEU RASCUNHO ANTERIOR:" + chr(10)
                        + json.dumps(saida, ensure_ascii=False) + chr(10) + chr(10)
                        + "CORRIJA APENAS ISTO e devolva o MESMO JSON completo:" + chr(10)
                        + chr(10).join(f"- {e}" for e in erros))
            resposta = gerar([{"type": "text", "text": correcao}], max_tokens=700)
            obj["resposta_bruta_retry"] = resposta
            saida = extrair_json(resposta)
        obj["alt_bruto"] = saida["alt_text"]
        obj["alt_text"] = pos_processar_alt(saida["alt_text"])
        obj["descricao_objeto"] = capitalizar_frases(saida["descricao_objeto"])
        flags_modelo = [f for f in saida.get("flags", [])
                        if f.get("tipo") not in ("artefato_estudio", "metadado_suspeito")]
        obj["flags"] = (obj["flags_artefato"] + obj["flags_registro"] + obj["flags_cor"]
                        + flags_modelo)
        obj["json_valido"] = True
        falhas_seguidas = []
    except Exception as e:
        obj["erro"] = repr(e)
        obj["alt_bruto"] = obj["alt_text"] = ""
        obj["descricao_objeto"] = ""
        obj["flags"] = obj["flags_registro"] + obj["flags_cor"]
        obj["json_valido"] = False
        falhas_seguidas.append(obj)
    obj["segundos_redacao"] = round(time.time() - inicio, 1)
    marca_retry = " (retry)" if obj["retry"] else ""
    if not obj["json_valido"]:
        print(f"[{n}/{len(objetos)}] {obj['titulo']} FALHOU {obj.get('erro', '')[:60]} | "
              f"bruta: {(obj.get('resposta_bruta') or '')[:90]!r}")
    else:
        print(f"[{n}/{len(objetos)}] {obj['titulo']}{marca_retry}: {obj['alt_text'][:70]}...")
    liberar_memoria()
    parar_se_quebrado(falhas_seguidas, CAMPOS_REDACAO, "redação")
    salvar_checkpoint()

print(f"{chr(10)}retries: {sum(1 for o in objetos if o.get('retry'))}/{len(objetos)} | "
      f"falhas: {sum(1 for o in objetos if not o['json_valido'])} | "
      f"pós-processamento de fundo agiu em "
      f"{sum(1 for o in objetos if o.get('alt_bruto') != o['alt_text'])} alts")

## Etapa 7 — A régua do repositório

A verificação automática **não está embutida** neste notebook: ela é baixada de `avaliacao/checar_lote.py`, no repositório público, no momento da execução. A razão é um achado do próprio projeto — a cópia da régua embutida no Notebook 04 ficou defasada quando a régua foi recalibrada depois de uma rodada. Fonte única: o notebook e a análise posterior medem com o mesmo código.

O placar sai separado por conjunto. O que aparece aqui é só a leitura imediata; a medição oficial é a do `avaliacao/rodar.py`, que roda sobre os arquivos salvos.

In [ ]:
# Etapa 7 — A régua vem do repositório: fonte única, sempre na versão atual
REPO_RAW = "https://raw.githubusercontent.com/eduardotosto/acervo-que-fala/main"
regua_src = buscar(f"{REPO_RAW}/avaliacao/checar_lote.py").text
REGUA = {"__name__": "checar_lote"}
exec(compile(regua_src, "checar_lote.py", "exec"), REGUA)
verificar = REGUA["verificar"]
print("régua carregada do repositório ✓ (checar_lote.py)\n")

for obj in objetos:
    obj["problemas"] = verificar(obj)
    if not obj["resolucao_ok"]:
        print(f"{obj['conjunto']:7} {obj['id']} {obj['titulo'][:28]:28} ⚑ falta_de_resolucao (sem texto, por política)")
        continue
    detalhe = "; ".join(f"{k}{' (' + v + ')' if v else ''}" for k, v in obj["problemas"])
    print(f"{obj['conjunto']:7} {obj['id']} {obj['titulo'][:28]:28} " + ("✓" if not obj["problemas"] else "⚠ " + detalhe))


def placar(nome, itens):
    if not itens:
        return
    sem = sum(1 for o in itens if not o["problemas"])
    media = sum(len(o["problemas"]) for o in itens) / len(itens)
    tipos = collections.Counter(f["tipo"] for o in itens for f in o["flags"])
    chaves = collections.Counter(k for o in itens for k, _ in o["problemas"])
    print(f"\n{nome}: {sem}/{len(itens)} sem problemas | {media:.1f} problemas por item | "
          f"retries {sum(1 for o in itens if o.get('retry'))} | "
          f"falhas de geração {sum(1 for o in itens if not o['json_valido'])}")
    print(f"  flags por tipo: {dict(tipos)}")
    print(f"  problemas por checagem: {dict(chaves)}")


casos = [o for o in objetos if o["conjunto"] == "casos"]
placar("CASOS (40)", casos)
placar("CASOS sem os 5 vistos no desenvolvimento (35)",
       [o for o in casos if not o["visto_no_desenvolvimento"]])
placar("HOLDOUT (10)", [o for o in objetos if o["conjunto"] == "holdout"])

## Etapa 8 — Salvar no Drive: dois arquivos

`06_lote_casos.json` (40 casos) e `06_lote_holdout.json` (10 casos), cada um com os prompts, a versão da rubrica, as versões das bibliotecas e o tempo de geração embutidos — o resultado carrega tudo o que é preciso para saber como foi produzido.

In [ ]:
# Etapa 8 — Salvar no Drive: um arquivo por conjunto
CAMPOS_SALVOS = ["id", "titulo", "conjunto", "visto_no_desenvolvimento", "registro",
                 "observacao", "enquadramento", "enquadramento_ok", "artefatos_obs",
                 "resolucao", "resolucao_ok", "contagem_registro", "retry", "artefatos_secao",
                 "marca_atribuicao", "flags_cor", "escala", "contradicao", "alt_bruto",
                 "alt_text", "descricao_objeto", "flags", "flags_registro", "diretrizes_usadas",
                 "json_valido", "problemas", "erro", "erro_observacao", "resposta_bruta",
                 "resposta_bruta_retry", "segundos_observacao", "segundos_contradicao",
                 "segundos_redacao"]
PADROES = {"retry": False, "contradicao": None, "erro": "", "erro_observacao": "",
           "resposta_bruta": "", "resposta_bruta_retry": "", "alt_bruto": "",
           "contagem_registro": "", "segundos_observacao": 0, "segundos_contradicao": 0,
           "segundos_redacao": 0}

for conjunto in ("casos", "holdout"):
    itens = [o for o in objetos if o["conjunto"] == conjunto]
    segundos = sum(o.get(c, 0) for o in itens
                   for c in ("segundos_observacao", "segundos_contradicao", "segundos_redacao"))
    resultado = {
        "notebook": VERSAO_NOTEBOOK,
        "conjunto": conjunto,
        "executado_em_utc": time.strftime("%Y-%m-%d %H:%M", time.gmtime()),
        "modelo": MODELO,
        "embedding": "Qwen/Qwen3-Embedding-0.6B (CPU)",
        "rubrica_versao": rubrica["versao"],
        "rag": "híbrido: diretriz da categoria pelo registro + glossário por similaridade (k=2)",
        "versoes_ambiente": VERSOES_AMBIENTE,
        "minutos_de_geracao": round(segundos / 60, 1),
        "prompt_observacao_v3": PROMPT_OBSERVACAO_V3,
        "prompt_redacao_v13": PROMPT_REDACAO_V13,
        "prompt_contradicao": PROMPT_CONTRADICAO,
        "tetos_categoria": TETOS_CATEGORIA,
        "itens": [{c: o.get(c, PADROES.get(c)) for c in CAMPOS_SALVOS} for o in itens],
    }
    destino = f"{PROJETO}/resultados/06_lote_{conjunto}.json"
    with open(destino, "w", encoding="utf-8") as f:
        json.dump(resultado, f, ensure_ascii=False, indent=2)
    print(f"salvo no Drive ✓  {destino}  ({len(itens)} itens, {resultado['minutos_de_geracao']} min de geração)")

---

## Fim — o que fazer agora

Avise o Claude que o Notebook 06 terminou. Ele busca os dois arquivos no Drive e roda `avaliacao/rodar.py`, que imprime as métricas do conjunto fixo.

O holdout rodou aqui pela primeira e única vez: o sistema não muda depois desta rodada. O que as métricas mostrarem — inclusive os defeitos — é o resultado do projeto.